# Raspberry Pi Camera Server — SD Card Recovery Runbook

**Purpose:** If the microSD card in the `cameraglasses` Raspberry Pi Zero W
ever gets corrupted, dies, or is lost, this notebook is a complete,
step-by-step rebuild from a blank card back to a fully working camera
server + physical shutdown button — exactly matching the last known-good
configuration.

**Scope of what gets rebuilt:**
- Raspberry Pi OS, flashed headless (Wi-Fi + SSH pre-configured)
- Camera dependencies (Picamera2 / libcamera)
- GPIO dependencies (gpiozero / lgpio) for the shutdown button
- `app.py` — the Flask camera server (JPEG, 1024×1024 capture)
- `shutdown_button.py` + `shutdown_service.py` — the GPIO3 shutdown button,
  running as its **own** systemd service, independent of the camera server
- Two systemd services (`camserver`, `shutdown-button`) with the
  `WorkingDirectory` fix that was required for the GPIO backend to load
- Passwordless-sudo rule needed for the shutdown button to work headlessly

Every command below is meant to be copy-pasted into an SSH session on the
Pi, one block at a time, in order. Read the explanation above each block
before running it.

## 0. Before you start: what you need

- A blank/formattable microSD card (8 GB+; the one already in the Pi is
  presumed dead/corrupted)
- A computer with an SD card reader and the **Raspberry Pi Imager** app
  installed (https://www.raspberrypi.com/software/)
- Your Wi-Fi network name and password
- This notebook, open on that computer (not on the Pi — the Pi has no
  working OS yet at this point)

**Known values used throughout this runbook** (adjust if yours differ):

| Setting | Value |
|---|---|
| Hostname | `cameraglasses` |
| Username | `hasaanhamid` |
| Camera server port | `5000` |
| GPIO shutdown button pin | GPIO3 / physical pin 5 (to GND on physical pin 6) |
| Capture resolution | 1024×1024 (matched to FastVLM 0.5B's fixed input size) |

## 1. Flash the SD card with Raspberry Pi OS (headless)

1. Insert the blank microSD card into your computer and open **Raspberry Pi Imager**.
2. Choose **Device**: Raspberry Pi Zero W.
3. Choose **OS**: Raspberry Pi OS (64-bit isn't supported on the Zero W's
   ARMv6 chip — use **Raspberry Pi OS (Legacy, 32-bit)** or the current
   default 32-bit image if offered for this device).
4. Choose **Storage**: your SD card.
5. Click the **gear/settings icon** (or press `Ctrl+Shift+X`) to open
   advanced options *before* writing, and set:
   - Hostname: `cameraglasses`
   - Enable SSH → **Use password authentication**
   - Username: `hasaanhamid`, and set your password
   - Configure Wi-Fi: your SSID + password, correct Wi-Fi country code
   - Locale/timezone: your local settings
6. Click **Save**, then **Write**, and confirm. This takes several minutes.
7. Once done, move the SD card into the Pi Zero W and power it on. Give it
   1–2 minutes for first boot.

## 2. Find the Pi and SSH in

From your computer (on the same Wi-Fi network):

```bash
ssh hasaanhamid@cameraglasses.local
```

If `.local` mDNS resolution doesn't work on your network, find the IP from
your router's connected-devices list instead and use:

```bash
ssh hasaanhamid@<PI_IP_ADDRESS>
```

Accept the host key prompt on first connect, and enter the password you
set in Raspberry Pi Imager.

## 3. Update the system

```bash
sudo apt update && sudo apt full-upgrade -y
sudo reboot
```

Wait ~30 seconds and SSH back in after the reboot.

## 4. Install camera dependencies

```bash
sudo apt install -y python3-picamera2 libcamera-apps
```

Confirm the camera is physically detected:

```bash
libcamera-hello --list-cameras || rpicam-hello --list-cameras
```

You should see your camera module listed with its sensor name (e.g.
`imx219` or `ov5647`). If nothing is listed, power off, reseat the CSI
ribbon cable (contacts facing the correct direction per your Pi model),
and try again before continuing.

## 5. Install GPIO dependencies (for the shutdown button)

```bash
sudo apt install -y python3-gpiozero python3-lgpio
```

Confirm your user is in the `gpio` group (it should be, by default, on
Raspberry Pi OS — but check anyway):

```bash
groups hasaanhamid
```

`gpio` should appear in the list. If it's missing:

```bash
sudo usermod -aG gpio hasaanhamid
sudo reboot
```

## 6. Create the Python virtual environment for the camera server

The venv is created with `--system-site-packages` so it can see the
apt-installed `picamera2`, `gpiozero`, and `lgpio` bindings, which can't be
cleanly `pip install`-ed standalone.

```bash
python3 -m venv --system-site-packages ~/camserver-env
source ~/camserver-env/bin/activate
pip install --upgrade pip
pip install flask pillow
```

## 7. Recreate `app.py` — the camera server

This is the JPEG version (WebP was tried and rolled back). Resolution
defaults are 1920×1080, but the systemd service in step 10 overrides this
to 1024×1024 via environment variables, matching FastVLM 0.5B's fixed
input size and avoiding wasted Wi-Fi bandwidth on unused pixels.

```bash
cat > ~/app.py << 'EOF'
#!/usr/bin/env python3
"""
Flask camera server for Raspberry Pi Zero (libcamera / Picamera2).
  GET /latest_hash_frame  -> low-res grayscale JPEG, buffered, fast poll (~150ms)
  GET /capture             -> high-res JPEG, on-demand shutter
  GET /status               -> JSON health / temp / uptime
"""

import io
import os
import time
import logging
import threading
import subprocess
from datetime import datetime

from flask import Flask, Response, jsonify
from PIL import Image
from picamera2 import Picamera2

# ---- logging ----------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
)

# ---- config (override via env vars, no code edits needed) -----------------
PORT          = int(os.environ.get("CAM_PORT", 5000))
LORES_W       = int(os.environ.get("CAM_LORES_W", 320))   # keep multiple of 32
LORES_H       = int(os.environ.get("CAM_LORES_H", 240))
MAIN_W        = int(os.environ.get("CAM_MAIN_W", 1920))
MAIN_H        = int(os.environ.get("CAM_MAIN_H", 1080))
LORES_QUALITY = int(os.environ.get("CAM_LORES_QUALITY", 70))
MAIN_QUALITY  = int(os.environ.get("CAM_MAIN_QUALITY", 90))
LORES_FPS_CAP = float(os.environ.get("CAM_LORES_FPS_CAP", 12))

# ---- camera: one Picamera2 instance, two concurrent streams ---------------
picam2 = Picamera2()
video_config = picam2.create_video_configuration(
    # NOTE: Picamera2's "RGB888" format is actually B,G,R byte order.
    # "BGR888" is the one that comes out as true R,G,B - needed for PIL.
    main={"size": (MAIN_W, MAIN_H), "format": "BGR888"},
    lores={"size": (LORES_W, LORES_H), "format": "YUV420"},
)
picam2.configure(video_config)
picam2.start()
time.sleep(1)  # let AE/AWB settle

camera_lock = threading.Lock()   # serializes calls into picam2
frame_lock  = threading.Lock()   # guards the shared lores jpeg buffer

latest_lores_jpeg = None
latest_lores_ts   = 0.0
lores_frame_count = 0
start_time = time.time()

app = Flask(__name__)


def _encode_gray_jpeg(y_plane, quality):
    img = Image.fromarray(y_plane, mode="L")
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    return buf.getvalue()


def lores_loop():
    """Background thread: keeps latest_lores_jpeg fresh. HTTP requests never
    touch the camera directly for /latest_hash_frame - they just read this buffer."""
    global latest_lores_jpeg, latest_lores_ts, lores_frame_count
    min_interval = 1.0 / LORES_FPS_CAP if LORES_FPS_CAP > 0 else 0

    while True:
        loop_start = time.time()
        try:
            with camera_lock:
                yuv = picam2.capture_array("lores")
            # YUV420: first LORES_H rows are the Y (luma) plane == grayscale already
            y_plane = yuv[:LORES_H, :LORES_W]
            jpeg_bytes = _encode_gray_jpeg(y_plane, LORES_QUALITY)
            with frame_lock:
                latest_lores_jpeg = jpeg_bytes
                latest_lores_ts = time.time()
                lores_frame_count += 1
        except Exception as e:
            logging.error("[lores_loop] %s", e)
            time.sleep(0.2)

        elapsed = time.time() - loop_start
        if min_interval > elapsed:
            time.sleep(min_interval - elapsed)


threading.Thread(target=lores_loop, daemon=True).start()


@app.route("/latest_hash_frame", methods=["GET"])
def latest_hash_frame():
    with frame_lock:
        jpeg_bytes, ts = latest_lores_jpeg, latest_lores_ts
    if jpeg_bytes is None:
        return jsonify({"error": "no frame yet"}), 503
    resp = Response(jpeg_bytes, mimetype="image/jpeg")
    resp.headers["X-Frame-Timestamp"] = str(ts)
    resp.headers["Cache-Control"] = "no-store"
    return resp


@app.route("/capture", methods=["GET"])
def capture():
    try:
        with camera_lock:
            rgb = picam2.capture_array("main")
        img = Image.fromarray(rgb, mode="RGB")
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=MAIN_QUALITY)
        jpeg_bytes = buf.getvalue()
    except Exception as e:
        return jsonify({"error": f"capture failed: {e}"}), 500
    resp = Response(jpeg_bytes, mimetype="image/jpeg")
    resp.headers["Cache-Control"] = "no-store"
    return resp


def get_cpu_temp_c():
    try:
        with open("/sys/class/thermal/thermal_zone0/temp") as f:
            return round(int(f.read().strip()) / 1000.0, 1)
    except Exception:
        pass
    try:
        out = subprocess.check_output(["vcgencmd", "measure_temp"]).decode()
        return float(out.strip().replace("temp=", "").replace("'C", ""))
    except Exception:
        return None


@app.route("/status", methods=["GET"])
def status():
    with frame_lock:
        ts, count = latest_lores_ts, lores_frame_count
    now = time.time()
    return jsonify({
        "status": "ok" if (now - ts) < 5 else "stale",
        "uptime_seconds": round(now - start_time, 1),
        "cpu_temp_c": get_cpu_temp_c(),
        "camera": {
            "lores_size": [LORES_W, LORES_H],
            "main_size": [MAIN_W, MAIN_H],
            "lores_frames_captured": count,
            "seconds_since_last_lores_frame": round(now - ts, 2) if ts else None,
        },
        "timestamp": datetime.utcnow().isoformat() + "Z",
    })


if __name__ == "__main__":
    logging.info("Starting camera server on 0.0.0.0:%d", PORT)
    app.run(host="0.0.0.0", port=PORT, threaded=True)
EOF
```

## 8. Wire the physical shutdown button (hardware step)

With the Pi powered off:

- One leg of a momentary push button → **physical pin 5** (GPIO3)
- Other leg → **physical pin 6** (GND), right next to it on the header

This uses the Pi's internal pull-up resistor (configured in software, not
wiring) — released = GPIO3 HIGH, pressed = GPIO3 LOW.

> **Wiring gotcha we hit before:** the two legs must be on the *button*,
> not wired directly to each other. A permanent short from pin 5 to pin 6
> with no button in between leaves GPIO3 stuck LOW forever, and the
> software never sees a press because it's watching for a HIGH→LOW
> *transition*, not a static LOW state.

## 9. Recreate `shutdown_button.py`

```bash
cat > ~/shutdown_button.py << 'EOF'
#!/usr/bin/env python3
"""
Isolated GPIO3 shutdown-button handler.

Hardware:
    GPIO3 (BCM3 / physical pin 5) --- momentary button --- GND (physical pin 6)
    Internal pull-up enabled: released = HIGH, pressed = LOW.

This module ONLY watches for a button press and issues a clean OS shutdown.
It does NOT and cannot implement wake-on-GPIO3 - that is a Raspberry Pi
hardware/firmware feature that operates only while the OS is not running.
No Python code can run during that state, so there is nothing to add here
for the wake side.
"""

import logging
import subprocess
import threading

from gpiozero import Button

logger = logging.getLogger("shutdown_button")

BUTTON_BCM_PIN = 3       # GPIO3 / physical pin 5
DEBOUNCE_SECONDS = 0.2   # ignore further edges within this window


class ShutdownButton:
    """
    Watches GPIO3 for a press and triggers a clean OS shutdown exactly once.

    Safety properties:
      - GPIO3 is only ever configured as an INPUT with the internal pull-up
        enabled (pull_up=True below). It is never driven as an output.
      - gpiozero/lgpio does edge-detection + debouncing internally, so this
        is event-driven - no busy-wait polling loop is used.
      - A lock-protected one-shot flag guarantees `shutdown -h now` is
        executed at most once, even if the pin bounces or the button is
        held down for a while.
    """

    def __init__(self, pin: int = BUTTON_BCM_PIN):
        self._lock = threading.Lock()
        self._shutdown_triggered = False

        # pull_up=True -> Broadcom internal pull-up resistor enabled.
        # bounce_time  -> software debounce window (seconds).
        self._button = Button(pin, pull_up=True, bounce_time=DEBOUNCE_SECONDS)
        self._button.when_pressed = self._on_press
        logger.info("Shutdown button armed on GPIO%d (physical pin 5)", pin)

    def _on_press(self):
        # Only the first press wins; every later call is a no-op.
        with self._lock:
            if self._shutdown_triggered:
                return
            self._shutdown_triggered = True

        logger.warning("Shutdown button pressed")
        self._do_shutdown()

    def _do_shutdown(self):
        try:
            subprocess.run(["sudo", "shutdown", "-h", "now"], check=True)
        except Exception as e:
            logger.error("Failed to execute shutdown command: %s", e)
            # Let a future press retry, since the shutdown itself didn't happen.
            with self._lock:
                self._shutdown_triggered = False

    def close(self):
        """Release GPIO resources cleanly (called on process exit)."""
        try:
            self._button.close()
            logger.info("Shutdown button GPIO released")
        except Exception as e:
            logger.error("Error releasing shutdown button GPIO: %s", e)
EOF
```

## 10. Recreate `shutdown_service.py`

This is the standalone entrypoint that keeps the button working
independently of the camera server, so a camera crash never disables
shutdown capability.

```bash
cat > ~/shutdown_service.py << 'EOF'
#!/usr/bin/env python3
"""
Standalone entrypoint that arms the GPIO3 shutdown button and then just
waits, independent of the camera server. Runs as its own systemd service
so the physical shutdown button keeps working even if app.py (the camera
server) is crashed, restarting, or being debugged manually.
"""

import logging
import signal

from shutdown_button import ShutdownButton

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
)

button = ShutdownButton()


def _handle_termination(signum, frame):
    logging.info("Received signal %s, releasing GPIO and exiting", signum)
    button.close()
    raise SystemExit(0)


signal.signal(signal.SIGTERM, _handle_termination)
signal.signal(signal.SIGINT, _handle_termination)

logging.info("Shutdown-button service ready, waiting for button presses")
signal.pause()  # sleep until a signal arrives; the Button callback runs from
                 # gpiozero's own background thread, not from this main thread
EOF
```

## 11. Passwordless sudo rule for shutdown

The button service calls `sudo shutdown -h now` from a non-interactive
background thread, so it needs to run without a password prompt:

```bash
echo "hasaanhamid ALL=(ALL) NOPASSWD: /sbin/shutdown, /usr/sbin/shutdown" | sudo tee /etc/sudoers.d/010_hasaanhamid-shutdown
sudo chmod 0440 /etc/sudoers.d/010_hasaanhamid-shutdown
```

Sanity check the file (this should NOT prompt for a password if it's set
up correctly — but don't actually run this test command yet, since it
will shut the Pi down for real; just confirm the file contents look
right):

```bash
sudo cat /etc/sudoers.d/010_hasaanhamid-shutdown
```

## 12. systemd service: `camserver`

`WorkingDirectory` matters here — without it, some libraries default to an
unwritable working directory under systemd and fail in ways that don't
happen when run manually from an interactive shell. `CAM_MAIN_W` /
`CAM_MAIN_H` set the 1024×1024 capture resolution.

```bash
cat > ~/camserver.service << 'EOF'
[Unit]
Description=Camera Server
After=network.target

[Service]
WorkingDirectory=/home/hasaanhamid
Environment=CAM_MAIN_W=1024
Environment=CAM_MAIN_H=1024
ExecStart=/home/hasaanhamid/camserver-env/bin/python3 /home/hasaanhamid/app.py
Restart=always
User=hasaanhamid

[Install]
WantedBy=multi-user.target
EOF
sudo mv ~/camserver.service /etc/systemd/system/camserver.service
```

## 13. systemd service: `shutdown-button`

Runs independently of `camserver`, using plain system Python (it only
needs `gpiozero`, not Flask/Picamera2/Pillow).

```bash
cat > ~/shutdown-button.service << 'EOF'
[Unit]
Description=GPIO3 Shutdown Button
After=multi-user.target

[Service]
WorkingDirectory=/home/hasaanhamid
ExecStart=/usr/bin/python3 /home/hasaanhamid/shutdown_service.py
Restart=always
User=hasaanhamid

[Install]
WantedBy=multi-user.target
EOF
sudo mv ~/shutdown-button.service /etc/systemd/system/shutdown-button.service
```

## 14. Enable and start both services

```bash
sudo systemctl daemon-reload
sudo systemctl enable --now camserver
sudo systemctl enable --now shutdown-button
sudo systemctl status camserver
sudo systemctl status shutdown-button
```

Both should show `Active: active (running)`. If either shows `failed`,
check `journalctl -u <service-name> -n 50 --no-pager` for the traceback
before moving on.

## 15. Verify the camera server end-to-end

```bash
curl -s http://localhost:5000/status
curl -s http://localhost:5000/latest_hash_frame -o /tmp/test_lores.jpg
curl -s http://localhost:5000/capture -o /tmp/test_full.jpg
```

Then confirm the capture resolution actually came out as 1024×1024:

```bash
source ~/camserver-env/bin/activate
python3 -c "from PIL import Image; print(Image.open('/tmp/test_full.jpg').size)"
```

Expected output: `(1024, 1024)`.

## 16. Verify the shutdown button (this WILL power off the Pi)

Only do this once everything above looks healthy, and you're ready to
physically power-cycle the Pi afterward.

```bash
journalctl -u shutdown-button -f
```

Leave that running, then physically press the button. You should see
`Shutdown button pressed` in the log, and the Pi should shut down and your
SSH session should drop — that's success.

Once it's powered down (all LEDs settled), pulling GPIO3 low again via the
**same button** should power it back on — this is a Raspberry Pi
hardware/firmware feature (GPIO3 wake), not something implemented in
software, so there's nothing further to configure for it.

## 17. Reboot test — confirm everything survives a restart

```bash
sudo reboot
```

SSH back in after ~30–60 seconds, then:

```bash
sudo systemctl status camserver
sudo systemctl status shutdown-button
```

Both should already be `active (running)` without you starting anything
manually. This confirms the whole stack — camera server, shutdown button,
and their systemd services — will survive future reboots and power
interruptions without any manual intervention.

**Recovery complete.**

## 18. For next time: make recovery even faster

Consider doing one or both of these so a future SD card failure takes
minutes, not an hour:

- **Keep a copy of `app.py`, `shutdown_button.py`, `shutdown_service.py`,
  and the two `.service` files off the Pi** (e.g. in a small git repo on
  your own computer), so you're copy-pasting from your own files instead
  of re-deriving them from this notebook each time.
- **Image the working SD card** once everything is confirmed stable, using
  Raspberry Pi Imager's "Use custom image" backup workflow or `dd` from
  another Linux machine, so a full restore is a single flash instead of
  this whole runbook.